# Quantification Module — Uncertainty Decomposition Demo

This notebook demonstrates the `UncertaintyDecomposer` class in `quantification_uncertainty.py`.

It propagates four independent uncertainty sources through the CEMF+IME flow-rate estimator:

| Source | 1σ | Method |
|---|---|---|
| Wind speed (ERA5 → plume layer) | ±20% | Literature prior (Varon 2021) |
| Sensitivity coefficient ε | ±15% | Literature prior (Varon 2021) |
| Plume mask threshold p* | ±Z% | Bootstrap over log-spaced p* range |
| Background annulus | ±W% | Jackknife N/S/E/W quadrants |
| **Combined** | ±tot% | Monte Carlo 10k, independent sources |

**Full run requires:** `data/npy_cache/<scene_id>.npy` + `results_bitemporal/<site>/original_<scene_id>.tif`  
**Physics methods** (CEMF, MC propagation) are demonstrated below with synthetic arrays — no data files needed.

**Prerequisites:** `numpy` only (Pillow/rasterio optional for full pipeline run).

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root.name not in ('methane-api', '') and repo_root != repo_root.parent:
    repo_root = repo_root.parent
scripts_dir = repo_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

print(f'Repo root: {repo_root}')

In [ ]:
import numpy as np

from quantification.quantification_uncertainty import (
    UncertaintyDecomposer,
    DETECTIONS,
    SIGMA_WIND_PCT,
    SIGMA_COEFF_PCT,
    SENSITIVITY_COEFF,
)

print(f'Detections registered  : {len(DETECTIONS)}')
print(f'σ_wind  (literature)   : ±{SIGMA_WIND_PCT}%')
print(f'σ_coeff (literature)   : ±{SIGMA_COEFF_PCT}%')
print(f'ε (sensitivity)        : {SENSITIVITY_COEFF}')

---
## Part 1 — CEMF Physics (pure numpy, no data files)

We inject a synthetic methane plume into a 100×100 pixel crop to demonstrate the physics:
- B12 is **enhanced** in the plume region (simulating the positive ΔR_B12 enhancement signal)
- All other conditions are uniform background

In [ ]:
rng = np.random.default_rng(42)
H, W = 100, 100

# Uniform background
b11 = np.full((H, W), 0.15, dtype=np.float32) + rng.standard_normal((H, W)).astype(np.float32) * 0.003
b12_bg = np.full((H, W), 0.12, dtype=np.float32) + rng.standard_normal((H, W)).astype(np.float32) * 0.003

# Inject a synthetic plume enhancement in the central 10×10 px region
b12_plume = b12_bg.copy()
b12_plume[40:50, 45:55] *= 1.08   # +8% B12 enhancement (positive ΔR signal)

mask = np.zeros((H, W), dtype=np.uint8)
mask[40:50, 45:55] = 1

d = UncertaintyDecomposer()
result = d.CemfFlowRate(b11, b12_plume, mask, wind_ms=3.0)

print(f"CEMF result:")
print(f"  flow_kgh        = {result['flow_kgh']:.2f} kg/h")
print(f"  n_plume_pixels  = {result['n_plume_pixels']}")
print(f"  plume_length_m  = {result['plume_length_m']:.1f} m")
print(f"  valid           = {result['valid']}")

### 1b. Wind sensitivity — how does flow scale with wind speed?

In [ ]:
wind_speeds = [1.0, 2.0, 3.0, 5.0, 8.0, 12.0]
print(f"{'wind (m/s)':>12}  {'flow (kg/h)':>12}  {'ratio to 3 m/s':>16}")
print('-' * 45)
ref_flow = d.CemfFlowRate(b11, b12_plume, mask, wind_ms=3.0)['flow_kgh']
for w in wind_speeds:
    r = d.CemfFlowRate(b11, b12_plume, mask, wind_ms=w)
    ratio = r['flow_kgh'] / ref_flow if ref_flow > 0 else float('nan')
    print(f"{w:>12.1f}  {r['flow_kgh']:>12.2f}  {ratio:>16.3f}")

---
## Part 2 — Monte Carlo Combined Uncertainty

Propagate all four uncertainty sources through Q = Q₀ × f_wind × f_coeff × f_mask × f_bg.

In [ ]:
# Simulate realistic sigma values from a real detection
base_flow    = 1071.0   # Bełchatów 2024-07-10 reported flow
sigma_mask   = 28.5     # typical bootstrap value
sigma_bg     = 12.3     # typical jackknife value

mc = d.MonteCarloCombined(
    base_flow_kgh   = base_flow,
    sigma_wind_pct  = SIGMA_WIND_PCT,
    sigma_coeff_pct = SIGMA_COEFF_PCT,
    sigma_mask_pct  = sigma_mask,
    sigma_bg_pct    = sigma_bg,
    n_samples       = 10_000,
)

print(f"Monte Carlo combined uncertainty (Q₀ = {base_flow:.0f} kg/h)")
print(f"  n_samples           : {mc['n_samples']:,}")
print(f"  mean                : {mc['mean_kgh']:.1f} kg/h")
print(f"  std                 : {mc['std_kgh']:.1f} kg/h")
print(f"  90% CI (p5–p95)     : [{mc['p5_kgh']:.1f}, {mc['p95_kgh']:.1f}] kg/h")
print(f"  σ_combined (MC)     : {mc['sigma_pct_mc']:.1f}%")
print(f"  σ_combined (quad)   : {mc['sigma_pct_quadrature']:.1f}%")
print(f"  Canonical ±30%      : [{base_flow*0.7:.0f}, {base_flow*1.3:.0f}] kg/h")

### 2b. How does total uncertainty change as mask bootstrap σ varies?

In [ ]:
print(f"{'σ_mask (%)':>12}  {'σ_combined_MC (%)':>20}  {'σ_quadrature (%)':>20}")
print('-' * 56)
for sm in [5, 15, 25, 40, 60]:
    mc_s = d.MonteCarloCombined(
        base_flow_kgh=1000.0, sigma_wind_pct=20.0, sigma_coeff_pct=15.0,
        sigma_mask_pct=sm, sigma_bg_pct=12.0, n_samples=5_000
    )
    print(f"{sm:>12}  {mc_s['sigma_pct_mc']:>20.1f}  {mc_s['sigma_pct_quadrature']:>20.1f}")

---
## Part 3 — Full Pipeline Run

Run the complete 4-source decomposition on all registered detections.
Detections whose `.npy` / `.tif` files are not present will return `status='error'` — this is expected in environments without the full data cache.

In [ ]:
print('Detection registry:')
for det in DETECTIONS:
    npy_ok = (repo_root / det['npy']).exists()
    tif_ok = (repo_root / det['tif']).exists()
    status = '✓ ready' if (npy_ok and tif_ok) else f'✗ missing: {"npy" if not npy_ok else "tif"}'
    print(f"  {det['label']:<40} {status}")

In [ ]:
decomposer = UncertaintyDecomposer(n_mc=10_000)
results    = decomposer.Run()

ok_results  = [r for r in results if r.get('status') == 'ok']
err_results = [r for r in results if r.get('status') != 'ok']

print(f'Completed: {len(ok_results)} ok, {len(err_results)} errors')
for r in err_results:
    print(f'  ✗ {r["label"]}: {r["error"][:80]}')

In [ ]:
# Print summary table for any successful detections
if ok_results:
    decomposer.PrintSummary(ok_results)
else:
    print('No successful detections in this environment.')
    print('To run the full pipeline: ensure data/npy_cache/ and results_bitemporal/ are populated.')

In [ ]:
# Save results (includes both ok and error records for audit trail)
decomposer.SaveResults(results)
print('Results saved to results_analysis/uncertainty_decomposition.json')

---
## Part 4 — Inspect existing results

If `results_analysis/uncertainty_decomposition.json` already exists from a previous full run,
load and inspect it directly.

In [ ]:
import json

existing_path = repo_root / 'results_analysis' / 'uncertainty_decomposition.json'
if existing_path.exists():
    existing = json.loads(existing_path.read_text())
    print(f'Loaded {len(existing)} records from existing results.\n')
    print(f"{'Label':<45} {'Status':>8} {'σ_wind':>8} {'σ_mask':>8} {'σ_MC':>8}")
    print('-' * 80)
    for r in existing:
        if r.get('status') == 'ok':
            unc = r['uncertainty']
            mc  = unc['combined_mc']
            sm  = unc['mask'].get('sigma_pct', float('nan'))
            sw  = unc['wind']['sigma_pct']
            print(f"{r['label']:<45} {'ok':>8} {sw:>7.1f}% "
                  f"{sm if isinstance(sm, str) else f'{sm:.1f}%':>8} {mc['sigma_pct_mc']:>7.1f}%")
        else:
            print(f"{r['label']:<45} {'error':>8}")
else:
    print('No existing results file found. Run Part 3 first with full data available.')